In [1]:
import torch

In [7]:
shape = (3,3)

t1 = torch.ones(shape)
t2 = torch.rand_like(t1, dtype=torch.float)

t1,t2

(tensor([[1., 1., 1.],
         [1., 1., 1.],
         [1., 1., 1.]]),
 tensor([[0.6475, 0.4286, 0.4817],
         [0.2134, 0.1831, 0.3888],
         [0.4171, 0.9204, 0.5587]]))

In [10]:
print(f"Shape: {t1.shape}")
print(f"Device: {t1.device}")
print(f"Datatype: {t1.dtype}")

Shape: torch.Size([3, 3])
Device: cpu
Datatype: torch.float32


In [11]:
#autograd is the automatic gradient descent (changes weights according to the direction that would lead to a local minima)
#Keeps track of what happens to the tensor

x_data = torch.tensor([[1.,2.], [3.,4.]])
x_data = torch.tensor([[1.], [3.]], requires_grad=True)

In [12]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(4.0, requires_grad=True)

y = a + b 
z = x*y
print(z)

tensor(20., grad_fn=<MulBackward0>)


In [ ]:
# .grad_fn is a breadcrumb that tells you how a tensor was made, None means it was directly created
# z was created by multiplication
print(f"grad_fn for z: {z.grad_fn}")

# y was created by addition
print(f"grad_fn for y: {y.grad_fn}")

# a was created by user and not by an operation
print(f"grad_fn for a: {a.grad_fn}")

# loss.backward() will use these trail of breadcrumbs to get the operations

grad_fn for z: <MulBackward0 object at 0x000001AE6DD5FA00>
grad_fn for y: <AddBackward0 object at 0x000001AE6DD5F310>
grad_fn for a: None


In [ ]:
# * is element wise multiplications, which takes the same size and multiplies the coefficients.

a = torch.tensor([[1,2], [3,4]])
b = torch.tensor([[10,20], [30,40]])
c = a *b
c
#note how it is 1*10, 2*20, 3*30, 4*40

tensor([[ 10,  40],
        [ 90, 160]])

In [ ]:
# @ is matric multiplcation. This is  2x3 @ 3x2 matrix
a = torch.tensor([[1,2], [3,4], [5,6]])
b = torch.tensor([[10,20, 30], [30,40, 60]])

c = a @ b
c

tensor([[ 70, 100, 150],
        [150, 220, 330],
        [230, 340, 510]])

In [ ]:
# reduction is any operation that reduces tensor to a smaller number of elements
a = torch.tensor([[10., 20., 30.],
                  [40.,50.,60.]])
a.mean()

#dim = 0 collapes the rows
#dim = 1 collapses columns
print(a.mean(dim=0)) # matrix is 2x3. collpasing rows makes it 1x3. So a column of values is one index

print(a.mean(dim=1)) # matrix is 2x3, collapsing columns makes it 2x1. So a row of values is one index


tensor([25., 35., 45.])
tensor([20., 50.])


In [26]:
# selecting data
x = torch.arange(12).reshape(3,4)
print(x)

col_2 = x[:, 2] #gets the third column which is index 2
print(col_2)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([ 2,  6, 10])


In [ ]:
#argmax gets the index of the highest number 
scores = torch.tensor([[10, 0, 5, 20, 1],
                      [1, 30, 2, 5, 0]])

best_indices = torch.argmax(scores, dim=1)
best_indices

tensor([3, 1])

In [ ]:
# torch.gather, get specific information. torch.gather is an optimized operation instead of a for loop
# Allows for advaanced dynamic capabilities

data = torch.tensor([
    [10, 11, 12, 13],
    [20, 21, 22, 23],
    [30, 31, 32, 33]
])

indices_to_select = torch.tensor([[2], [0], [3]])
print(torch.gather(data, dim=1, index=indices_to_select))

tensor([[12],
        [20],
        [33]])


In [ ]:
# # step 1 of 5. Forward pass, model's first guess. Simple Linear Regression. y = XW + b 
# x = input
# w = weights
# b = bias

N = 10

D_in = 1
D_out = 1

X = torch.randn(N, D_in)

true_W = torch.tensor([[2.0]])
true_b = torch.tensor(1.0)
y_true = X @ true_W + true_b + torch.randn(N, D_out) * .1 # little noise
# model will never see true_W or true b. will have to discover it itself

W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad= True)

print(f"Initial Weight W: \n {W} \n")
print(f"Initial Bias b: \n {b} \n")

y = X @ W + b

y[:3],y_true[:3] 

Initial Weight W: 
 tensor([[0.2768]], requires_grad=True) 

Initial Bias b: 
 tensor([1.1546], requires_grad=True) 



(tensor([[1.3336],
         [1.2471],
         [0.9056]], grad_fn=<SliceBackward0>),
 tensor([[ 2.3565],
         [ 1.6881],
         [-0.8442]]))

In [49]:
# step 2 backward Pass. Forward pass is guess, backward pass is the autopsy. Using mean squared error to determine the error rate
error = y - y_true
squared_error = error **2
loss = squared_error.mean()
loss #loss also has a breadcrumb trail

#single command to get Pytorch to use. calculate all gradients from loss from true.
#gets the gradient of loss with respect to W and gradient of loss with respect to b
loss.backward() # python performs most critical aspect. Gets the .grad attribute
print(f"Gradient for W: \n {W.grad}")
print(f"Gradient for b: \n {b.grad}")


Gradient for W: 
 tensor([[-2.4678]])
Gradient for b: 
 tensor([1.0224])


In [60]:
# adjusting weights, training loop. Gradient descent. t1 = t0 - n Gradient(loss). t is parameters, n (eta) is learning rate, 
# torch.no_grad() don't track parameter updates
# .grad.zero_() reset gradients at each iteration
eta, epochs = .01, 100
W = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad= True)

#training loop
for epoch in range(epochs):
    # forward pass and loss calculated
    y = X @ W + b
    y_true = X @ true_W + true_b
    loss = torch.mean((y - y_true)**2)
    
    # backward pass
    loss.backward()
    
    #update weights and bias
    with torch.no_grad():
        W -= eta*W.grad; b -= eta * b.grad
    
    # reset gradient values so it can recalculate for next loop
    W.grad.zero_(); b.grad.zero_()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item()}, W = {W.item()}, b = {b.item()}")
    
print(f"Weights: {W.item()}")
print(f"Bias: {b.item()}")

Epoch 0, Loss = 0.07024337351322174, W = 1.7794197797775269, b = 1.150996208190918
Epoch 10, Loss = 0.04711543023586273, W = 1.8119361400604248, b = 1.1166950464248657
Epoch 20, Loss = 0.031799860298633575, W = 1.8392908573150635, b = 1.0896472930908203
Epoch 30, Loss = 0.021612588316202164, W = 1.8623582124710083, b = 1.0683714151382446
Epoch 40, Loss = 0.014801813289523125, W = 1.881856918334961, b = 1.0516834259033203
Epoch 50, Loss = 0.010221772827208042, W = 1.8983790874481201, b = 1.0386369228363037
Epoch 60, Loss = 0.007121688220649958, W = 1.912412405014038, b = 1.028476357460022
Epoch 70, Loss = 0.005008061416447163, W = 1.924359917640686, b = 1.0205992460250854
Epoch 80, Loss = 0.0035554971545934677, W = 1.9345555305480957, b = 1.0145251750946045
Epoch 90, Loss = 0.002548655029386282, W = 1.9432759284973145, b = 1.0098713636398315
Weights: 1.9500538110733032
Bias: 1.0066444873809814


In [ ]:
# torch.nn module
#prebuilt layers that froms backbone of leanrning algorithms
# torch.NN.linear gives us linear regression. Packages parameters in objects
# Parameters are a special tensor that has requires_grad= True by default
# Auto-registers with the model
#handles bookkeeping
D_in = 1
D_out = 1
linear_layer = torch.nn.Linear(in_features= D_in, out_features= D_out)
print(f"Layer's Weight: {linear_layer.weight} \n")
print(f"Layer's bias: {linear_layer.bias} \n")

y_nn = linear_layer(X)

print(f"Output of nn.Linear (first 3 rows): \n {y_nn[:3]}")

Layer's Weight: Parameter containing:
tensor([[-0.5973]], requires_grad=True) 

Layer's bias: Parameter containing:
tensor([-0.5470], requires_grad=True) 

Output of nn.Linear (first 3 rows): 
 tensor([[-0.9333],
        [-0.7466],
        [-0.0097]], grad_fn=<SliceBackward0>)


In [67]:
# Activation functions make it non-linear 
# NN.ReLu(), if input is negative, make it 0
# nn.GELU() is Gaussian error linear unit. A smoother, gently curving version of ReLu
# nn.softmax() convert logits (raw model scores) into probability distribution. 
softmax = torch.nn.Softmax(dim=-1)
logits = torch.tensor([[1.0, 3.0, 0.5, 1.5], [-1.0, 2.0, 1.0, 0.0]])
probabilities = softmax(logits)
print(f"Output probabilities: \n {probabilities}\n")
print(f"Sum probabilities: {probabilities[0].sum()}\n")

Output probabilities: 
 tensor([[0.0939, 0.6942, 0.0570, 0.1549],
        [0.0321, 0.6439, 0.2369, 0.0871]])

Sum probabilities: 1.0



In [81]:
#for building blocks of GPT, LLAMA and Gemini
#nn.embedding, turns words -> numbers 
vocab_size = 10 # 10 unique words
embedding_dim = 3 # Represent each word with a 3D vector

embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)

input_ids = torch.tensor([[1, 5, 0, 8]])
word_vectors = embedding_layer(input_ids)

print(word_vectors)
    

tensor([[[ 0.1229, -1.9413, -1.0472],
         [-0.9059,  0.0714,  0.8490],
         [-0.9934, -1.5408, -0.3783],
         [-0.0497, -0.1535,  0.2413]]], grad_fn=<EmbeddingBackward0>)


In [ ]:
#nn.layernorm. Normalizes ranges so that the mean should be around 0
norm_layer = torch.nn.LayerNorm(normalized_shape=3)
input_features = torch.tensor([[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]])
normalized_features = norm_layer(input_features)

print(normalized_features)
print(f"Mean : {normalized_features.mean(dim=-1)}")
print(f"Std Dev : {normalized_features.std(dim=-1)}")

tensor([[[-1.2247,  0.0000,  1.2247],
         [-1.2247,  0.0000,  1.2247]]], grad_fn=<NativeLayerNormBackward0>)
Mean : tensor([[0., 0.]], grad_fn=<MeanBackward1>)
Std Dev : tensor([[1.2247, 1.2247]], grad_fn=<StdBackward0>)


In [ ]:
#nn.dropout prevents overfitting by taking out outliers. Randomly zeros neruons during trainings  
drop_out = torch.nn.Dropout(p =.5)
input_tensor = torch.ones(1, 10)

#activate dropout for training
drop_out.train()
output_during_train = drop_out(input_tensor)

#deactivate dropout for pevaluation and prediction
drop_out.eval()
output_during_eval = drop_out(input_tensor)

# Half the numbers are zeroed, the others are scaled up to match the valeus
print(f"Output during training: {output_during_train}")
print(f"Output during evaluation: {output_during_eval}")

Output during training: tensor([[0., 2., 2., 2., 0., 0., 0., 0., 2., 2.]])
Output during evaluation: tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])


In [ ]:
# nn.Module to organize module (instruction booklet and baseplate) for structure and how bricks connect
# torch.optim adjusts all parameters according to the instructions from gradients
# clean standard and scalable

#blueprint
# 1. inherit torch.nn.Module
# 2. __init__: define layers
# 3. forward: connect layers

#creation of a linear regression model
import torch.nn as nn
class LinearRegressionModel(nn.Module):
    #these will define layers of a neural network
    # each line of code represents a 
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear_layer = nn.Linear(in_features, out_features)
    
    #connects layers / defines how to calculate output as input for the next layer
    #
    def forward(self, x):
        return self.linear_layer(x)
    

linReg = LinearRegressionModel(in_features= 1, out_features= 1)
print(linReg)

LinearRegressionModel(
  (linear_layer): Linear(in_features=1, out_features=1, bias=True)
)


In [90]:
# creates gradient descent for learning algorithm
import torch.optim as optim

learning_rate = 0.01

optimizer = optim.Adam(linReg.parameters(), lr=learning_rate)

loss_fn = nn.MSELoss() # calculates loss function as Mean Squared Error loss

In [ ]:
#training loop, 3 line mantra

# 1.
# optimizer.zero_grad()

# #2
# loss.backward()

# #3
# optimizer.step()

#final training loop

epochs = 100

for epoch in range(epochs):
    y = linReg(X)
    
    # y_true are the expected outputs whereas y is the model's output
    loss = loss_fn(y, y_true)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epch {epoch}, Loss = {loss.item()}")


print(linReg.linear_layer.weight.item())
print(linReg.linear_layer.bias.item())

Epch 0, Loss = 1.8559377125415555e-12
Epch 10, Loss = 1.8559377125415555e-12
Epch 20, Loss = 1.8559377125415555e-12
Epch 30, Loss = 1.8559377125415555e-12
Epch 40, Loss = 1.8559377125415555e-12
Epch 50, Loss = 1.8559377125415555e-12
Epch 60, Loss = 1.9976908582214348e-12
Epch 70, Loss = 1.6790125062851602e-12
Epch 80, Loss = 1.6790125062851602e-12
Epch 90, Loss = 1.6790125062851602e-12
1.9999983310699463
0.9999997019767761


5 steps of neural network
1. Create model / layers
2. calculate loss function
3. optimizer.zero_grad()
4. loss.backward()
5. optimizer.step()